In [46]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE

In [3]:
df = pd.read_csv("../data/raw/creditcard.csv")
df.shape

(284807, 31)

In [4]:
X = df.drop("Class", axis = 1)
y = df["Class"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [9]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=1,
    class_weight="balanced_subsample"
)

rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(

In [10]:
y_prob = rf_model.predict_proba(X_test)[:,1]

threshold = 0.7
y_pred = (y_prob >= threshold).astype(int)

In [11]:
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.84      0.78      0.81        98

    accuracy                           1.00     56962
   macro avg       0.92      0.89      0.90     56962
weighted avg       1.00      1.00      1.00     56962

[[56850    14]
 [   22    76]]


In [12]:
results = X_test.copy()
results["ActualClass"] = y_test.values
results["FraudProbability"] = y_prob
results["RiskScore"] = np.round(y_prob*100).astype(int)
results["PredictedClass"] = y_pred
 

In [13]:
def score_to_band(score:int) -> str:
    if score < 20:
        return "Low"
    elif score <=70:
        return "Medium"
    else:
        return "High"

results["RiskBand"] = results["RiskScore"].apply(score_to_band)

In [15]:
flagged = results[results["PredictedClass"] == 1].copy()
flagged.shape

flagged = flagged.sort_values("RiskScore", ascending=False)
flagged.head(10)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V25,V26,V27,V28,Amount,ActualClass,FraudProbability,RiskScore,PredictedClass,RiskBand
77348,57007.0,-1.271244,2.462675,-2.851395,2.324480,-1.372245,-0.948196,-3.065234,1.166927,-2.268771,...,0.224228,0.756335,0.632800,0.250187,0.01,1,0.999713,100,1,High
119781,75581.0,-2.866364,2.346949,-4.053307,3.983359,-3.463186,-1.280953,-4.474764,1.216655,-2.309829,...,-0.506901,-0.371741,0.615257,0.803163,124.53,1,0.999534,100,1,High
229712,146022.0,0.908637,2.849024,-5.647343,6.009415,0.216656,-2.397014,-1.819308,0.338527,-2.819883,...,0.465058,0.210510,0.648705,0.360224,1.18,1,0.999769,100,1,High
153835,100298.0,-22.341889,15.536133,-22.865228,7.043374,-14.183129,-0.463145,-28.215112,-14.607791,-9.481456,...,0.447154,-0.632816,-4.380154,-0.467863,1.00,1,0.999374,100,1,High
42696,41203.0,-8.426814,6.241659,-9.946470,8.199614,-8.213093,-2.522046,-11.643028,5.339500,-7.051016,...,0.467594,0.483162,1.195671,0.198294,88.23,1,0.999592,100,1,High
15810,27252.0,-25.942434,14.601998,-27.368650,6.378395,-19.104033,-4.684806,-18.261393,17.052566,-3.742605,...,1.820378,-0.219359,1.388786,0.406810,99.99,1,0.999667,100,1,High
141260,84204.0,-1.927453,1.827621,-7.019495,5.348303,-2.739188,-2.107219,-5.015848,1.205868,-4.382713,...,0.718717,1.111151,1.277707,0.819081,512.25,1,0.999495,100,1,High
151519,95628.0,-17.518909,12.572118,-19.038538,11.190895,-13.554721,-0.411924,-23.189397,-5.301412,-8.630390,...,-0.720128,-0.232603,-3.021992,-0.478158,1.63,1,0.999683,100,1,High
143333,85285.0,-7.030308,3.421991,-9.525072,5.270891,-4.024630,-2.865682,-6.989195,3.791551,-4.622730,...,0.353634,1.042458,1.359516,-0.272188,0.00,1,0.999777,100,1,High
154719,102671.0,-4.991758,5.213340,-9.111326,8.431986,-3.435516,-1.827565,-7.114303,3.431207,-3.875643,...,-0.174617,0.406703,-0.402339,-0.882886,0.00,1,0.999667,100,1,High


In [29]:
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index = X_train.columns
).sort_values(ascending=False)

feature_importance.head(10)

V14    0.192044
V10    0.113576
V12    0.099999
V4     0.094237
V17    0.089993
V3     0.068241
V11    0.057792
V16    0.040306
V2     0.037645
V9     0.027283
dtype: float64

In [30]:
top_features = feature_importance.head(5).index.tolist()
top_features

['V14', 'V10', 'V12', 'V4', 'V17']

In [38]:
def explain_transaction(row):
    reasons = []

    if row["RiskScore"] >= 90:
        reasons.append("Very high fraud risk score")
    elif row["RiskScore"] >= 70:
        reasons.append("Elevated fraud risk score")

    if row["Amount"] > 200:
        reasons.append("High transaction amount")
    elif row["Amount"] < 1:
        reasons.append("Very small transaction amount")
    
    for feature in top_features:
        if abs(row[feature]) >3:
            reasons.append(f"Strong anomaly in {feature}")
    
    if not reasons:
        reasons.append("Unusuall overall feature pattern")

    return "Flagged due to " + ", ".join(reasons) + ".2"

In [32]:
flagged["Explanation"] = flagged.apply(explain_transaction, axis=1)

In [34]:
flagged[[
    "Amount",
    "RiskScore",
    "RiskBand",
    "ActualClass",
    "PredictedClass",
    "Explanation"
]].head(15)

,Amount,RiskScore,RiskBand,ActualClass,PredictedClass,Explanation
77348,0.01,100,High,1,1,"Flagged due to Very high fraud risk score, Ver..."
119781,124.53,100,High,1,1,"Flagged due to Very high fraud risk score, Str..."
229712,1.18,100,High,1,1,"Flagged due to Very high fraud risk score, Str..."
153835,1.00,100,High,1,1,"Flagged due to Very high fraud risk score, Str..."
42696,88.23,100,High,1,1,"Flagged due to Very high fraud risk score, Str..."
15810,99.99,100,High,1,1,"Flagged due to Very high fraud risk score, Str..."
141260,512.25,100,High,1,1,"Flagged due to Very high fraud risk score, Hig..."
151519,1.63,100,High,1,1,"Flagged due to Very high fraud risk score, Str..."
143333,0.00,100,High,1,1,"Flagged due to Very high fraud risk score, Ver..."
154719,0.00,100,High,1,1,"Flagged due to Very high fraud risk score, Ver..."


In [35]:
true_positives = flagged[flagged["ActualClass"] == 1].copy()

true_positives[[
    "Amount",
    "RiskScore",
    "RiskBand",
    "Explanation"
]].head(10)

,Amount,RiskScore,RiskBand,Explanation
77348,0.01,100,High,"Flagged due to Very high fraud risk score, Ver..."
119781,124.53,100,High,"Flagged due to Very high fraud risk score, Str..."
229712,1.18,100,High,"Flagged due to Very high fraud risk score, Str..."
153835,1.00,100,High,"Flagged due to Very high fraud risk score, Str..."
42696,88.23,100,High,"Flagged due to Very high fraud risk score, Str..."
15810,99.99,100,High,"Flagged due to Very high fraud risk score, Str..."
141260,512.25,100,High,"Flagged due to Very high fraud risk score, Hig..."
151519,1.63,100,High,"Flagged due to Very high fraud risk score, Str..."
143333,0.00,100,High,"Flagged due to Very high fraud risk score, Ver..."
154719,0.00,100,High,"Flagged due to Very high fraud risk score, Ver..."


In [36]:
false_positives = flagged[flagged["ActualClass"] == 0].copy()

false_positives[[
    "Amount",
    "RiskScore",
    "RiskBand",
    "Explanation"
]].head(10)

,Amount,RiskScore,RiskBand,Explanation
16110,1.00,99,High,"Flagged due to Very high fraud risk score, Str..."
14920,1.00,97,High,"Flagged due to Very high fraud risk score, Str..."
17592,89.99,94,High,"Flagged due to Very high fraud risk score, Str..."
19145,89.99,92,High,"Flagged due to Very high fraud risk score, Str..."
18125,89.99,91,High,"Flagged due to Very high fraud risk score, Str..."
18150,89.99,91,High,"Flagged due to Very high fraud risk score, Str..."
43432,89.99,91,High,"Flagged due to Very high fraud risk score, Str..."
10783,89.99,83,High,"Flagged due to Elevated fraud risk score, Stro..."
190263,0.76,82,High,"Flagged due to Elevated fraud risk score, Very..."
13942,89.99,76,High,"Flagged due to Elevated fraud risk score, Stro..."


Final Dataset

In [43]:
results["Explanation"] = ""
results.loc[flagged.index, "Explanation"] = flagged["Explanation"]

In [44]:
final_df = results.copy()

cols = [
    "Amount",
    "FraudProbability",
    "RiskScore",
    "RiskBand",
    "PredictedClass",
    "ActualClass",
    "Explanation"
] + [col for col in final_df.columns  if col.startswith("V")]

final_df = final_df[cols]

In [45]:
flagged_df = final_df[final_df["PredictedClass"] == 1].copy()

In [47]:
os.makedirs("../data/processed", exist_ok=True)
final_df.to_csv("../data/processed/fraud_scored_full.csv", index=False)
flagged_df.to_csv("../data/processed/fraud_flagged_only.csv", index=False)